# deepfakekiller — CNN/ViT head fine-tuning (Colab)

`deepfakekiller.py`의 `model_cnn`(EfficientNet-V2-S)과 `model_trans`(ViT-B-16)을 real/fake 이미지 데이터로 fine-tuning해서
`weights_cnn.pth`, `weights_vit.pth`를 만듭니다. 데이터셋과 학습은 모두 이 노트북(즉 클라우드) 안에서만 일어나고,
로컬 맥북으로는 학습이 끝난 뒤 결과물(`weights_cnn.pth`, `weights_vit.pth`, 수백MB 수준)만 다운로드하면 됩니다.

**사용법 — 그냥 `런타임 > 모두 실행`으로 끝까지 돌려도 됩니다.**
1. Colab에서 이 파일을 열고 `런타임 > 런타임 유형 변경 > GPU`로 설정
2. `런타임 > 모두 실행`
3. 5번 섹션에서 `weights_cnn.pth`, `weights_vit.pth` 다운로드 후 로컬 `deepfakekiller/weights/` 폴더에 넣기
4. 6번(in-distribution)·7번(OOD) 벤치마크까지 돌려서 나온 표를 README.md의 결과표 자리에 붙여넣기

**ArtiFact(Kaggle) 인증 — 한 번만 하면 그다음부턴 자동입니다.**
kaggle.com/settings에서 API 토큰(kaggle.json)을 받아서, Google Drive의 `MyDrive/deepfakekiller/kaggle.json`에
한 번만 넣어두세요. 그러면 다음부터는 노트북이 Drive에서 자동으로 가져다 씁니다 (Drive 마운트 자체는
세션마다 승인 클릭 한 번 필요). 안 해도 됩니다 — 없으면 그냥 조용히 건너뛰고 `extra_hf_datasets`
(Defactify + OpenFake, 둘 다 인증 불필요)만으로 계속 진행합니다.

**최신 생성기(SD3/SDXL/DALL-E 3/MidJourney v6, Gemini) 대응**: 기존 ArtiFact 데이터셋은 2023년 이전 생성기
위주라 최신 생성기 이미지를 잘 못 잡는다는 게 확인된 상태입니다 (README "알려진 한계" 참고). 이걸 메우려고
설정 셀의 `extra_hf_datasets` **리스트**에 데이터셋을 추가하면 ArtiFact와 자동으로 병합해서 학습합니다 —
기본값으로 Defactify(SD3/SD2.1/SDXL/DALL-E 3/MidJourney v6)와 OpenFake의 `train` 스플릿(Gemini의
nano-banana/nano-banana-pro 포함)이 둘 다 들어있습니다. `resume_from`에 기존 `weights_cnn.pth`/
`weights_vit.pth` 경로를 넣으면 처음부터 다시 학습하지 않고 그 위에 이어서 fine-tuning합니다.
각 항목의 컬럼명/라벨 값은 실제로 huggingface.co에서 확인한 값으로 맞춰뒀습니다 — **다른 데이터셋을
추가할 경우에만** 그 데이터셋 페이지에서 컬럼명/라벨 값을 확인해서 새 항목을 리스트에 넣으면 됩니다.

**Gemini 포함 OOD 검증**: 7번 셀은 `ComplexDataLab/OpenFake`의 `core/test`(위 학습에 쓰는 `train`
스플릿과는 겹치지 않는 별도 held-out)로 진짜 일반화됐는지 검증합니다. 무료(CC-BY-SA-4.0), API 키 불필요.

**2026-08-24 1차 재학습 결과** (Defactify만 병합): in-distribution AUC 0.96, OOD(Gemini) AUC 0.63으로
거의 랜덤 수준 — 그래서 이번에 OpenFake의 `train` 스플릿도 학습에 추가했습니다. 자세한 건 README 참고.

**메모리 주의**: 스트리밍 데이터셋 셔플 버퍼(`buffer_size`)를 너무 크게 잡으면(예: 2000) 세션이
"RAM을 전부 써서" 죽을 수 있습니다 — 실제로 겪었던 문제라 지금은 200으로 낮춰뒀습니다. 그래도
죽으면 더 낮추세요.

In [ ]:
!pip install -q kagglehub datasets scikit-learn

# Hugging Face의 최신 "Xet" 다운로드 백엔드가 큰 파일에서 종종
# "File reconstruction error: Background writer channel closed"로 죽는
# 알려진 버그가 있어서, 예전 방식(HTTP)으로 강제 다운로드하도록 끈다.
import os
os.environ["HF_HUB_DISABLE_XET"] = "1"

# ---------------------------------------------
# 스트리밍 다운로드 워치독
# ---------------------------------------------
# 실제로 겪은 문제: OpenFake의 특정 parquet 샤드가 "peer closed connection" ->
# huggingface_hub 자체 재시도(5/5)까지 다 거친 뒤에도 예외를 던지지 않고 그대로
# 응답 없이 멈춰서, 1시간 반 넘게 GPU 세션 시간만 태운 적이 있다. huggingface_hub/
# datasets/pyarrow 중 정확히 어느 레이어가 멈추는지 특정하기 어렵고, 그 레이어가
# 예외를 안 던지면 아래 각 데이터 로딩 셀의 try/except로는 애초에 잡을 수가 없다.
# 그래서 signal.alarm으로 "이터레이터에서 항목 하나를 꺼내는 데 N초 넘게 걸리면
# 강제로 예외를 발생시키는" 워치독을 따로 둔다 — 이게 실제 방어선이고, 각 셀의
# try/except는 이 워치독이 던지는 예외(및 진짜 네트워크 예외)를 받아 지금까지
# 받은 부분 결과로 계속 진행하는 역할을 한다.
import signal

class StreamStallTimeout(Exception):
    pass

def _stall_alarm_handler(signum, frame):
    raise StreamStallTimeout("스트리밍 항목 하나를 받는 데 제한 시간을 넘겨서, 응답 없이 멈춘 것으로 판단해 중단했습니다.")

def iterate_with_timeout(iterable, timeout_sec):
    """이터레이터에서 항목 하나를 꺼내는 데 timeout_sec초 이상 걸리면 강제 중단시킨다.
    signal.alarm은 유닉스 전용(Colab=Linux 런타임에서는 정상 동작)이라, 혹시 다른
    환경에서 이 노트북을 그대로 돌리면 타임아웃 없이 원래 동작으로 안전하게 대체된다."""
    it = iter(iterable)
    has_alarm = hasattr(signal, "SIGALRM")
    old_handler = signal.signal(signal.SIGALRM, _stall_alarm_handler) if has_alarm else None
    try:
        while True:
            if has_alarm:
                signal.alarm(timeout_sec)
            try:
                item = next(it)
            except StopIteration:
                return
            finally:
                if has_alarm:
                    signal.alarm(0)
            yield item
    finally:
        if has_alarm and old_handler is not None:
            signal.signal(signal.SIGALRM, old_handler)

In [ ]:
# ---------------------------------------------
# 설정 (Config) — 데이터 소스/하이퍼파라미터를 여기서 바꿉니다
# ---------------------------------------------
CONFIG = {
    # "kaggle" | "huggingface" | "folder"
    "data_source": "kaggle",

    # data_source == "kaggle" 일 때 쓰이는 식별자. 실행 전 kaggle.com에서 존재 여부를 확인하세요.
    "kaggle_handle": "awsaf49/artifact-dataset",

    # kaggle.json을 Google Drive의 이 경로에 한 번 저장해두면, 다음 실행부터는
    # 자동으로 여기서 가져와서 ~/.kaggle/kaggle.json에 복사한다 (매번 업로드/재입력 불필요).
    # 처음 한 번은: drive.google.com에서 MyDrive에 "deepfakekiller" 폴더를 만들고
    # kaggle.com/settings에서 받은 kaggle.json을 그 안에 넣어두면 됨.
    "kaggle_json_drive_path": "/content/drive/MyDrive/deepfakekiller/kaggle.json",

    # data_source == "huggingface" 일 때 쓰이는 설정 (huggingface.co/datasets 에서 검색)
    "hf_dataset": "",
    "hf_image_col": "image",
    "hf_label_col": "label",   # 0 = real, 1 = fake 로 가정 (다르면 아래 매핑 로직 수정)

    # data_source == "folder" 일 때 쓰이는 경로. 이 안에 real/ 과 fake/ 하위 폴더가 있어야 함
    # (예: Google Drive를 마운트해서 "/content/drive/MyDrive/deepfake_data")
    "folder_path": "/content/data",

    # 첫 학습은 빠르게 돌려보기 위해 클래스당 샘플 수를 제한. 전체 데이터를 다 쓰려면 None
    "max_samples_per_class": 20000,

    "image_size": 224,
    "batch_size": 32,
    # 에포크별 성능 곡선(리포트용)을 보고 싶으면 넉넉하게(예: 10~15) 잡을 것.
    # warm-start라 5로도 동작은 하지만, 곡선이 의미 있으려면 점이 더 많아야 함.
    "epochs": 5,
    "lr_head": 1e-4,
    "lr_backbone": 1e-5,
    "val_split": 0.1,
    "seed": 42,
    "num_workers": 4,
    "output_dir": "/content/weights",

    # ── 스트리밍 다운로드 워치독 ──
    # HF Hub에서 스트리밍으로 이미지를 받아올 때, 항목 하나(next())를 꺼내는 데
    # 이 시간(초) 이상 걸리면 응답 없이 멈춘 것으로 보고 강제 중단한다 (1번 셀의
    # iterate_with_timeout 참고). 실제로 특정 parquet 샤드가 재시도 5/5까지 다
    # 거친 뒤에도 예외 없이 멈춰서 GPU 세션 시간을 1시간 반 넘게 태운 적이 있어서
    # 넣은 안전장치. 정상적인 이미지 하나 받는 데는 몇 초면 충분하니 90초면 넉넉함.
    "stream_stall_timeout_sec": 90,

    # ── 에포크별 리포트용 로깅 ──
    # val AUC는 학습 루프가 어차피 매 에포크 계산하던 값이라 공짜로 기록되고,
    # OOD(Gemini 등 held-out) AUC는 아래 두 값으로 몇 에포크마다 얼마나 잴지 조절한다.
    # 매 에포크 전체 OOD 세트(1000개)를 다 재는 건 느려서, 작은 캐시 샘플(기본 300개)을
    # 학습 시작 전에 한 번만 받아두고 그걸로 빠르게 잰다.
    "epoch_log_ood_every": 2,       # N 에포크마다 OOD AUC 측정 (1이면 매 에포크)
    "epoch_log_ood_samples": 300,   # 에포크별 체크용 소규모 OOD 샘플 수

    # ── 기존 체크포인트에 이어서 학습(warm-start) ──
    # 비워두면("") ImageNet 사전학습 backbone에서 처음부터 학습.
    # 기존 weights_cnn.pth/weights_vit.pth 경로를 넣으면 그 위에서 이어서 fine-tuning해서
    # ArtiFact 기반으로 이미 배운 걸 유지하면서 아래 최신 생성기 데이터만 추가로 학습한다.
    # 로컬에서 Drive로 올려서 경로를 넣거나, 노트북 파일 업로드 기능으로 세션에 직접 올려도 됨.
    # 2026-08-24 학습분(weights_cnn.pth/weights_vit.pth, 이미 Defactify까지 반영됨)을
    # 다시 Drive/업로드해서 여기 경로를 넣으면, 아래 OpenFake 추가분만큼만 더 이어서 배운다.
    "resume_from": {
        "cnn": "",   # 예: "/content/drive/MyDrive/deepfakekiller/weights_cnn.pth"
        "vit": "",
    },

    # ── 최신 생성기(SD3/SDXL/DALL-E 3/MidJourney v6, 그리고 Gemini) 데이터셋들을
    # ArtiFact와 병합 ──
    # 각 항목: dataset(HF 경로), config(서브셋 이름, 없으면 None), split,
    # image_col/label_col(실제 컬럼명), real_value(이 값이면 real, 아니면 fake로 이진화),
    # max_samples(클래스당 상한). 리스트에 항목을 추가/제거해서 여러 데이터셋을 동시에 병합 가능.
    #
    # 2026-08-24 1차 재학습(Defactify만 병합) 결과: in-distribution AUC 0.96인데
    # OOD(OpenFake core/test, Gemini nano-banana 포함, 학습에 안 씀) AUC는 0.63으로
    # 거의 랜덤 수준이었음 — "다른 최신 생성기로 배우면 Gemini에도 전이된다"는 가설이
    # 기각된 것. 그래서 Gemini 샘플 자체를 학습에도 넣기 위해 OpenFake의 *train*
    # 스플릿을 추가함 (7번 벤치마크 셀이 쓰는 core/*test*와는 겹치지 않는 별도
    # 스플릿이라 평가 데이터 누출 없음).
    "extra_hf_datasets": [
        {
            "dataset": "Rajarshi-Roy-research/Defactify_Image_Dataset",
            "config": None,
            "split": "train",
            "image_col": "Image",
            "label_col": "Label_A",
            "real_value": 0,
            "max_samples": 20000,
        },
        {
            "dataset": "ComplexDataLab/OpenFake",
            "config": "core",
            "split": "train",
            "image_col": "image",
            "label_col": "label",
            "real_value": "real",
            "max_samples": 20000,
        },
    ],

    # ── OOD(진짜 일반화) 검증 전용 — 학습에는 절대 안 쓰고 7번 벤치마크 셀에서만 사용 ──
    # ComplexDataLab/OpenFake의 core/test 스플릿은 "학습 때 못 본 생성기 + 못 본 real 이미지
    # 소스"로만 구성된 OOD 평가 전용 스플릿이고, Gemini의 이미지 생성 모델(nano-banana/
    # nano-banana-pro)이 포함돼 있어 README의 "Gemini 못 잡음" 한계를 직접 검증할 수 있다.
    # 무료(CC-BY-SA-4.0), API 키/유료 구독 불필요. 다만 proprietary 생성기(OpenAI/Google 등)
    # 쪽 fake 샘플은 제공사 라이선스상 비상업적 용도로만 재배포 가능 — 지금처럼 로컬에서
    # 평가용으로만 쓰는 건 문제없지만, 이 fake 이미지 자체를 상업적으로 재배포하면 안 됨.
    # 위 extra_hf_datasets에 넣은 건 OpenFake의 train 스플릿이라, 여기 test 스플릿과는
    # 겹치지 않는다 — 학습/평가 분리 유지됨. 4번 학습 루프의 에포크별 로깅도 이 설정을
    # 재사용해서 작은 캐시 샘플을 만든다.
    "ood_eval_dataset": "ComplexDataLab/OpenFake",
    "ood_eval_config": "core",
    "ood_eval_split": "test",
    "ood_eval_image_col": "image",
    "ood_eval_label_col": "label",
    "ood_eval_real_value": "real",   # 라벨이 0/1이 아니라 문자열 "real"/"fake"
    "ood_eval_max_samples": 1000,    # test 스플릿 전체를 다 안 받고 이 개수만 샘플링
}

import os
os.makedirs(CONFIG["output_dir"], exist_ok=True)
print(CONFIG)

In [ ]:
# ---------------------------------------------
# Kaggle 인증 확인 (data_source == "kaggle"일 때만 필요)
# ---------------------------------------------
# 순서: ① 로컬(~/.kaggle/kaggle.json 또는 환경변수)에 이미 있으면 그걸 씀
#      ② 없으면 Google Drive의 CONFIG["kaggle_json_drive_path"]를 확인해서
#         있으면 자동으로 복사해옴 (Drive 마운트는 세션당 한 번 승인 클릭 필요)
#      ③ Drive에도 없으면 안내만 하고 기다리지 않고 건너뜀 — 다음 데이터 확보
#         셀에서 ArtiFact 없이 extra_hf_datasets(Defactify + OpenFake, 인증 불필요)
#         만으로 계속 진행한다.
if CONFIG["data_source"] == "kaggle":
    import os
    from pathlib import Path

    kaggle_json_path = Path.home() / ".kaggle" / "kaggle.json"
    has_env = bool(os.environ.get("KAGGLE_USERNAME")) and bool(os.environ.get("KAGGLE_KEY"))
    KAGGLE_AUTH_OK = kaggle_json_path.exists() or has_env

    if KAGGLE_AUTH_OK:
        print("✅ Kaggle 인증 정보 확인됨 (로컬) — ArtiFact도 같이 받습니다.")
    else:
        drive_path = Path(CONFIG.get("kaggle_json_drive_path", ""))
        try:
            from google.colab import drive
            drive.mount("/content/drive")
        except Exception as e:
            print(f"⚠️ Drive 마운트 실패({e}) — Drive 확인은 건너뜁니다.")
            drive_path = None

        if drive_path is not None and drive_path.exists():
            kaggle_json_path.parent.mkdir(parents=True, exist_ok=True)
            kaggle_json_path.write_bytes(drive_path.read_bytes())
            kaggle_json_path.chmod(0o600)
            KAGGLE_AUTH_OK = True
            print(f"✅ Drive({drive_path})에서 kaggle.json 자동 복사 완료 — ArtiFact도 같이 받습니다.")
        else:
            KAGGLE_AUTH_OK = False
            print("⚠️ Kaggle 인증 정보가 로컬에도 Drive에도 없어서 ArtiFact(Kaggle)는 건너뜁니다.")
            print("   다음부터 자동으로 되게 하려면: kaggle.com/settings에서 kaggle.json을 받아")
            print(f"   Google Drive의 {CONFIG.get('kaggle_json_drive_path')} 에 한 번만 넣어두세요")
            print("   (drive.google.com에서 MyDrive에 deepfakekiller 폴더 만들고 그 안에 넣으면 됨).")
            print("   지금은 extra_hf_datasets(Defactify + OpenFake)만으로 계속 진행합니다.")
else:
    KAGGLE_AUTH_OK = True  # kaggle을 안 쓰는 설정이면 이 플래그는 의미 없음
    print("data_source가 kaggle이 아니라서 Kaggle 인증은 필요 없습니다.")

## 1. 데이터 확보

`data_source`에 따라 이미지 경로 리스트와 라벨(0=real, 1=fake)을 만듭니다.
ArtiFact 데이터셋처럼 각 폴더 안에 `metadata.csv`(컬럼: `image_path`, `target` — 0=real, 1=fake)가 있으면 그 라벨을 그대로 사용합니다.
metadata.csv가 없는 데이터셋(직접 올린 `folder` 방식 등)은 **경로에 `real`이라는 디렉토리명이 포함되어 있으면 real(0), 아니면 fake(1)**로 보는 휴리스틱으로 대체합니다.
데이터셋 구조가 다르면 아래 `load_from_root` / `is_real_path` 함수를 수정하세요.

In [ ]:
import random
from pathlib import Path
import pandas as pd

random.seed(CONFIG["seed"])

def is_real_path(path: Path) -> bool:
    parts = [p.lower() for p in path.parts]
    return any(p == "real" or p.startswith("real_") or p.startswith("real-") for p in parts)

def load_from_root(root: Path):
    """metadata.csv(ArtiFact 스타일: image_path, target 컬럼)가 있으면 그 라벨을 쓰고,
    없으면 폴더명에 'real'이 포함되는지로 판단하는 휴리스틱을 쓴다.
    target은 문서상 0=real/1=fake지만 실제로는 생성기별 ID가 섞여 있을 수 있어
    0이면 real(0), 그 외에는 전부 fake(1)로 강제 이진화한다."""
    paths, labels = [], []
    metadata_files = list(root.rglob("metadata.csv"))

    if metadata_files:
        print(f"metadata.csv {len(metadata_files)}개 발견 — csv 라벨 사용")
        raw_targets = set()
        for csv_path in metadata_files:
            try:
                df = pd.read_csv(csv_path)
            except Exception as e:
                print(f"  {csv_path} 읽기 실패: {e}")
                continue
            if "image_path" not in df.columns or "target" not in df.columns:
                print(f"  {csv_path}: image_path/target 컬럼 없음, 건너뜀")
                continue
            for row in df.itertuples(index=False):
                img_path = csv_path.parent / str(row.image_path)
                raw_target = int(row.target)
                raw_targets.add(raw_target)
                paths.append(str(img_path))
                labels.append(0 if raw_target == 0 else 1)

        if len(raw_targets) > 2:
            print(f"참고: target 원본 값이 {sorted(raw_targets)}로 2종류를 넘어서, "
                  f"0=real / 나머지=fake로 강제 이진화했습니다.")

        # 문서와 실제 구조가 다를 수 있으니 샘플 일부만 실제 존재 여부 확인
        sample = random.sample(paths, min(20, len(paths))) if paths else []
        missing = [p for p in sample if not Path(p).exists()]
        if missing:
            print(f"경고: 샘플 {len(sample)}개 중 {len(missing)}개 경로가 존재하지 않습니다. "
                  f"image_path가 csv 위치 기준 상대경로가 아닐 수 있습니다. 예: {missing[0]}")
    else:
        print("metadata.csv 없음 — 폴더명(real 포함 여부) 휴리스틱 사용")
        exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
        for p in root.rglob("*"):
            if p.suffix.lower() in exts:
                paths.append(str(p))
                labels.append(0 if is_real_path(p) else 1)

    return paths, labels

image_paths, labels = [], []

if CONFIG["data_source"] == "kaggle" and not globals().get("KAGGLE_AUTH_OK", True):
    print("Kaggle 인증이 없어서 ArtiFact는 건너뜁니다 (바로 위 인증 확인 셀 참고).")

elif CONFIG["data_source"] == "kaggle":
    import kagglehub
    dataset_root = Path(kagglehub.dataset_download(CONFIG["kaggle_handle"]))
    print(f"다운로드 완료: {dataset_root}")
    image_paths, labels = load_from_root(dataset_root)

elif CONFIG["data_source"] == "folder":
    image_paths, labels = load_from_root(Path(CONFIG["folder_path"]))

elif CONFIG["data_source"] == "huggingface":
    from datasets import load_dataset
    assert CONFIG["hf_dataset"], "CONFIG['hf_dataset']를 설정하세요 (예: 'org/dataset-name')"
    ds = load_dataset(CONFIG["hf_dataset"], split="train")

    hf_cache_dir = Path("/content/hf_images")
    hf_cache_dir.mkdir(exist_ok=True)
    for i, ex in enumerate(ds):
        img = ex[CONFIG["hf_image_col"]]
        raw_label = int(ex[CONFIG["hf_label_col"]])
        out_path = hf_cache_dir / f"{i}.jpg"
        img.convert("RGB").save(out_path, quality=95)
        image_paths.append(str(out_path))
        labels.append(0 if raw_label == 0 else 1)
else:
    raise ValueError(f"알 수 없는 data_source: {CONFIG['data_source']}")

print(f"기본 데이터셋({CONFIG['data_source']}): 총 {len(image_paths)}개")

# ---------------------------------------------
# 최신 생성기 데이터셋 병합 (extra_hf_datasets 리스트에 설정된 항목들, 여러 개 가능)
# ---------------------------------------------
import gc
from collections import defaultdict as _defaultdict

for src in CONFIG.get("extra_hf_datasets", []):
    from datasets import load_dataset as _load_extra

    name = src["dataset"]
    print(f"\n추가 데이터셋 로드 중: {name} ({src.get('config')}/{src['split']})")

    # streaming=True + take(): 데이터셋 전체(특히 OpenFake는 약 250만 장 규모)를
    # 통째로 받지 않고, 실제로 쓸 클래스당 상한(max_samples)의 몇 배 정도만
    # 스트리밍으로 가져온 뒤 거기서 클래스 균형을 맞춘다. 소스가 한쪽 클래스로
    # 심하게 치우쳐 있으면 배수(4배)로도 부족할 수 있으니, 아래 출력되는
    # real/fake 개수를 보고 필요하면 fetch_multiplier를 늘릴 것.
    #
    # buffer_size 주의: shuffle()의 버퍼는 "디코딩된 이미지"를 버퍼 크기만큼
    # 메모리에 들고 있는다. 이전에 2000으로 뒀다가 OpenFake(이미지가 큼) 처리 중
    # "세션이 RAM을 전부 써서 죽음" 크래시가 났었음 — 200으로 줄임. 그래도 죽으면
    # 더 줄일 것(예: 50~100).
    cap = src.get("max_samples")
    fetch_multiplier = 4
    fetch_n = cap * fetch_multiplier if cap is not None else None

    ds = _load_extra(name, src.get("config"), split=src["split"], streaming=True)
    if fetch_n is not None:
        ds = ds.shuffle(seed=CONFIG["seed"], buffer_size=200).take(fetch_n)

    cache_dir = Path(f"/content/extra_hf_images/{name.replace('/', '_')}")
    cache_dir.mkdir(parents=True, exist_ok=True)

    src_paths, src_labels = [], []
    try:
        for i, ex in enumerate(iterate_with_timeout(ds, CONFIG["stream_stall_timeout_sec"])):
            img = ex[src["image_col"]]
            raw_label = ex[src["label_col"]]
            out_path = cache_dir / f"{i}.jpg"
            img.convert("RGB").save(out_path, quality=95)
            src_paths.append(str(out_path))
            src_labels.append(0 if raw_label == src["real_value"] else 1)
    except Exception as e:
        # HF Hub의 큰 parquet 샤드는 가끔 다운로드 도중 연결이 끊기거나(재시도 5회
        # 후에도 실패), 심지어 예외 없이 그냥 응답 없이 멈추기도 한다(iterate_with_timeout
        # 워치독이 그 경우 StreamStallTimeout을 던짐). 여기서 안 잡으면 노트북 전체
        # (런타임 > 모두 실행)가 멈추므로, 그때까지 받은 만큼만으로 계속 진행한다 —
        # 어차피 streaming이라 fetch_n을 정확히 못 채워도 아래 클래스당 상한(cap)에서
        # 다시 걸러지므로 무해하다.
        print(f"⚠️ {name} 스트리밍 중 오류/멈춤으로 중단됨 ({e}). "
              f"지금까지 받은 {len(src_paths)}개로 계속 진행합니다.")

    # 클래스당 상한 적용 (뒤에 나오는 전체 max_samples_per_class와 별개로 여기서 먼저 제한)
    if cap is not None:
        by_label_src = _defaultdict(list)
        for p, l in zip(src_paths, src_labels):
            by_label_src[l].append(p)
        src_paths, src_labels = [], []
        for l, ps in by_label_src.items():
            random.shuffle(ps)
            ps = ps[:cap]
            src_paths.extend(ps)
            src_labels.extend([l] * len(ps))

    print(f"{name}: real {sum(1 for l in src_labels if l == 0)}개, "
          f"fake {sum(1 for l in src_labels if l == 1)}개")
    image_paths.extend(src_paths)
    labels.extend(src_labels)

    # 다음 데이터셋으로 넘어가기 전에 스트리밍 버퍼/캐시를 명시적으로 해제
    del ds
    gc.collect()

print(f"\n병합 후 총합: {len(image_paths)}개")
n_real = sum(1 for l in labels if l == 0)
n_fake = sum(1 for l in labels if l == 1)
print(f"real: {n_real}, fake: {n_fake}")
assert n_real > 0 and n_fake > 0, "real 또는 fake 이미지가 0개입니다. metadata.csv 경로 구조나 is_real_path 휴리스틱을 확인하세요."
assert n_real + n_fake == len(labels), "라벨이 0/1 이외의 값을 포함합니다."

In [ ]:
# 클래스당 샘플 수 제한 + train/val 분할
from collections import defaultdict

by_label = defaultdict(list)
for path, label in zip(image_paths, labels):
    by_label[label].append(path)

cap = CONFIG["max_samples_per_class"]
final_paths, final_labels = [], []
for label, paths in by_label.items():
    random.shuffle(paths)
    if cap is not None:
        paths = paths[:cap]
    final_paths.extend(paths)
    final_labels.extend([label] * len(paths))

combined = list(zip(final_paths, final_labels))
random.shuffle(combined)
final_paths, final_labels = zip(*combined)

n_val = int(len(final_paths) * CONFIG["val_split"])
val_paths, val_labels = final_paths[:n_val], final_labels[:n_val]
train_paths, train_labels = final_paths[n_val:], final_labels[n_val:]

print(f"train: {len(train_paths)}, val: {len(val_paths)}")

## 2. Dataset / DataLoader

`deepfakekiller.py`의 전처리(224x224, ImageNet 정규화)와 동일하게 맞춥니다.

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

preprocess = transforms.Compose([
    transforms.Resize((CONFIG["image_size"], CONFIG["image_size"])),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_augment = transforms.Compose([
    transforms.Resize((CONFIG["image_size"], CONFIG["image_size"])),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

class DeepfakeDataset(Dataset):
    def __init__(self, paths, labels, transform):
        self.paths = paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        try:
            img = Image.open(self.paths[idx]).convert("RGB")
        except Exception:
            img = Image.new("RGB", (CONFIG["image_size"], CONFIG["image_size"]))
        x = self.transform(img)
        y = torch.tensor([float(self.labels[idx])])
        return x, y

train_ds = DeepfakeDataset(train_paths, train_labels, train_augment)
val_ds = DeepfakeDataset(val_paths, val_labels, preprocess)

train_loader = DataLoader(train_ds, batch_size=CONFIG["batch_size"], shuffle=True,
                           num_workers=CONFIG["num_workers"], pin_memory=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=CONFIG["batch_size"], shuffle=False,
                         num_workers=CONFIG["num_workers"], pin_memory=True)

## 3. 모델 정의

`deepfakekiller.py`와 **가중치 key가 정확히 일치**하도록 동일한 구조로 만듭니다 (그래야 `load_state_dict`가 바로 됩니다).

In [ ]:
import torch.nn as nn
import torchvision.models as models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {device}")

model_cnn = models.efficientnet_v2_s(weights="DEFAULT")
num_ftrs_cnn = model_cnn.classifier[1].in_features
model_cnn.classifier[1] = nn.Linear(num_ftrs_cnn, 1)
model_cnn = model_cnn.to(device)

model_trans = models.vit_b_16(weights="DEFAULT")
num_ftrs_trans = model_trans.heads.head.in_features
model_trans.heads.head = nn.Linear(num_ftrs_trans, 1)
model_trans = model_trans.to(device)

# CONFIG["resume_from"]에 경로가 지정돼 있으면 ImageNet 사전학습 대신
# 기존 fine-tuning 체크포인트 위에서 이어서 학습한다 (warm-start).
def _maybe_resume(model, path, name):
    if path:
        state = torch.load(path, map_location=device)
        model.load_state_dict(state)
        print(f"✅ {name}: 기존 체크포인트에서 이어서 학습 ({path})")
    return model

model_cnn = _maybe_resume(model_cnn, CONFIG.get("resume_from", {}).get("cnn"), "CNN")
model_trans = _maybe_resume(model_trans, CONFIG.get("resume_from", {}).get("vit"), "ViT")

print("OK")

## 4. 학습 루프

두 모델을 동시에 학습(같은 배치를 한 번에 통과)하고, backbone은 낮은 LR로, 마지막 head는 높은 LR로 학습합니다 (discriminative LR).
각 epoch마다 val AUC를 측정해 가장 좋은 시점의 가중치를 메모리에 보관합니다.

In [ ]:
import os
import copy
import csv
import numpy as np
from sklearn.metrics import roc_auc_score, accuracy_score

os.makedirs(CONFIG["output_dir"], exist_ok=True)
cnn_ckpt_path = os.path.join(CONFIG["output_dir"], "weights_cnn.pth")
vit_ckpt_path = os.path.join(CONFIG["output_dir"], "weights_vit.pth")

def make_optimizer(model, lr_head, lr_backbone, head_module):
    head_params = list(head_module.parameters())
    head_ids = {id(p) for p in head_params}
    backbone_params = [p for p in model.parameters() if id(p) not in head_ids]
    return torch.optim.AdamW([
        {"params": backbone_params, "lr": lr_backbone},
        {"params": head_params, "lr": lr_head},
    ])

opt_cnn = make_optimizer(model_cnn, CONFIG["lr_head"], CONFIG["lr_backbone"], model_cnn.classifier[1])
opt_trans = make_optimizer(model_trans, CONFIG["lr_head"], CONFIG["lr_backbone"], model_trans.heads.head)

criterion = nn.BCEWithLogitsLoss()
scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))

best_auc_cnn, best_state_cnn = -1.0, None
best_auc_trans, best_state_trans = -1.0, None

# ---------------------------------------------
# 리포트용 에포크별 로그 준비
# ---------------------------------------------
# val AUC는 아래 루프가 어차피 매 에포크 계산하던 값이라 그냥 기록만 하면 되고,
# OOD(Gemini 등 held-out) AUC는 몇 에포크마다 재는 게 아니면 너무 느리다 — 그래서
# ood_eval_dataset에서 작은 샘플을 학습 시작 전에 한 번만 캐싱해두고, N 에포크마다
# 그 캐시로 CNN+ViT 앙상블 AUC만 빠르게 잰다(ELA는 신경망이 아니라 여기선 생략).
# buffer_size는 200으로 작게 유지 — shuffle()이 디코딩된 이미지를 그 개수만큼
# 메모리에 들고 있어서, 이전에 2000으로 뒀다가 OOM으로 세션이 죽은 적 있음.
epoch_log = []
OOD_EVAL_EVERY = CONFIG.get("epoch_log_ood_every", 2)
OOD_LOG_SAMPLES = CONFIG.get("epoch_log_ood_samples", 300)

ood_log_x, ood_log_y = None, None
if CONFIG.get("ood_eval_dataset"):
    print(f"에포크별 로깅용 OOD 샘플 {OOD_LOG_SAMPLES}개 미리 캐싱 중 "
          f"({CONFIG['ood_eval_dataset']} {CONFIG['ood_eval_config']}/{CONFIG['ood_eval_split']})...")
    from datasets import load_dataset as _load_ood_log

    _ood_log_ds = _load_ood_log(
        CONFIG["ood_eval_dataset"], CONFIG["ood_eval_config"],
        split=CONFIG["ood_eval_split"], streaming=True,
    ).shuffle(seed=CONFIG["seed"], buffer_size=200).take(OOD_LOG_SAMPLES)

    _xs, _ys = [], []
    try:
        for ex in iterate_with_timeout(_ood_log_ds, CONFIG["stream_stall_timeout_sec"]):
            img = ex[CONFIG["ood_eval_image_col"]].convert("RGB")
            raw_label = ex[CONFIG["ood_eval_label_col"]]
            _xs.append(preprocess(img))
            _ys.append(0 if raw_label == CONFIG["ood_eval_real_value"] else 1)
    except Exception as e:
        # HF Hub 스트리밍이 큰 샤드 다운로드 도중 네트워크 오류로 끊기거나, 예외 없이
        # 그냥 응답 없이 멈출 수도 있다(그 경우 iterate_with_timeout이 강제로 끊음).
        # 여기서 안 잡으면 학습 루프 전체가 시작도 못 하고 멈추므로, 그때까지 받은
        # 만큼만으로 계속 진행한다.
        print(f"⚠️ OOD 로깅용 샘플 캐싱 중 오류/멈춤으로 중단됨 ({e}). "
              f"지금까지 받은 {len(_xs)}개로 계속 진행합니다.")

    if _xs:
        ood_log_x = torch.stack(_xs).to(device)
        ood_log_y = np.array(_ys)
        print(f"캐싱 완료: {len(ood_log_y)}개 (real {int((ood_log_y == 0).sum())}, "
              f"fake {int((ood_log_y == 1).sum())})")
    else:
        print("OOD 로깅용 샘플을 하나도 못 받아서, 에포크별 OOD 체크는 건너뜁니다.")
    del _ood_log_ds, _xs, _ys


def _quick_ood_auc():
    """캐싱해둔 소규모 OOD 샘플로 CNN+ViT 앙상블 AUC만 빠르게 계산 (ELA는 생략).
    ood_log_x 전체(기본 300장)를 한 배치로 통과시키면 학습 배치 크기(기본 32)보다
    훨씬 큰 배치가 되어, 학습 중 이미 점유된 GPU 메모리와 겹쳐 CUDA OOM이 날 수 있다
    (실제로 겪은 문제는 아니지만 재현 가능한 위험이라 예방 차원에서 학습 배치 크기
    단위로 나눠서 처리한다)."""
    if ood_log_x is None:
        return None
    model_cnn.eval()
    model_trans.eval()
    bs = CONFIG["batch_size"]
    p_cnn_chunks, p_vit_chunks = [], []
    with torch.no_grad():
        for i in range(0, ood_log_x.shape[0], bs):
            chunk = ood_log_x[i:i + bs]
            p_cnn_chunks.append(torch.sigmoid(model_cnn(chunk)).cpu().numpy().ravel())
            p_vit_chunks.append(torch.sigmoid(model_trans(chunk)).cpu().numpy().ravel())
    p_cnn = np.concatenate(p_cnn_chunks)
    p_vit = np.concatenate(p_vit_chunks)
    ensemble = p_cnn * 0.45 + p_vit * 0.45
    try:
        return float(roc_auc_score(ood_log_y, ensemble))
    except ValueError:
        return None


for epoch in range(CONFIG["epochs"]):
    model_cnn.train()
    model_trans.train()
    running_loss = 0.0

    for x, y in train_loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)

        opt_cnn.zero_grad()
        opt_trans.zero_grad()
        with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
            out_cnn = model_cnn(x)
            out_trans = model_trans(x)
            loss = criterion(out_cnn, y) + criterion(out_trans, y)

        scaler.scale(loss).backward()
        scaler.step(opt_cnn)
        scaler.step(opt_trans)
        scaler.update()
        running_loss += loss.item()

    # --- validation ---
    model_cnn.eval()
    model_trans.eval()
    y_true, p_cnn, p_trans = [], [], []
    with torch.no_grad():
        for x, y in val_loader:
            x = x.to(device, non_blocking=True)
            p_cnn.extend(torch.sigmoid(model_cnn(x)).cpu().numpy().ravel().tolist())
            p_trans.extend(torch.sigmoid(model_trans(x)).cpu().numpy().ravel().tolist())
            y_true.extend(y.numpy().ravel().tolist())

    auc_cnn = roc_auc_score(y_true, p_cnn)
    auc_trans = roc_auc_score(y_true, p_trans)
    acc_cnn = accuracy_score(y_true, [1 if p > 0.5 else 0 for p in p_cnn])
    acc_trans = accuracy_score(y_true, [1 if p > 0.5 else 0 for p in p_trans])

    is_last_epoch = (epoch + 1) == CONFIG["epochs"]
    ood_auc = None
    if ood_log_x is not None and ((epoch + 1) % OOD_EVAL_EVERY == 0 or is_last_epoch):
        ood_auc = _quick_ood_auc()

    epoch_log.append({
        "epoch": epoch + 1,
        "train_loss": running_loss / len(train_loader),
        "val_auc_cnn": auc_cnn,
        "val_auc_vit": auc_trans,
        "ood_auc_ensemble": ood_auc,
    })

    ood_str = f" | OOD(Gemini 등) AUC={ood_auc:.4f}" if ood_auc is not None else ""
    print(f"[epoch {epoch+1}/{CONFIG['epochs']}] loss={running_loss/len(train_loader):.4f} "
          f"| CNN AUC={auc_cnn:.4f} acc={acc_cnn:.4f} "
          f"| ViT AUC={auc_trans:.4f} acc={acc_trans:.4f}{ood_str}")

    # 매 epoch 최고 기록 갱신 시 메모리뿐 아니라 디스크에도 바로 저장
    # (런타임 재시작/연결 끊김으로 메모리 변수가 날아가도 디스크의 /content 는
    #  커널만 재시작된 경우라면 보존되므로, 학습 결과를 잃지 않기 위한 안전장치)
    if auc_cnn > best_auc_cnn:
        best_auc_cnn = auc_cnn
        best_state_cnn = copy.deepcopy(model_cnn.state_dict())
        torch.save(best_state_cnn, cnn_ckpt_path)
    if auc_trans > best_auc_trans:
        best_auc_trans = auc_trans
        best_state_trans = copy.deepcopy(model_trans.state_dict())
        torch.save(best_state_trans, vit_ckpt_path)

print(f"\n최고 val AUC — CNN: {best_auc_cnn:.4f}, ViT: {best_auc_trans:.4f}")
print(f"체크포인트: {cnn_ckpt_path}, {vit_ckpt_path}")

# ---------------------------------------------
# 에포크 로그 저장 (표 + 그래프) — 리포트에 바로 쓸 수 있는 산출물
# ---------------------------------------------
log_csv_path = os.path.join(CONFIG["output_dir"], "epoch_log.csv")
with open(log_csv_path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["epoch", "train_loss", "val_auc_cnn", "val_auc_vit", "ood_auc_ensemble"])
    writer.writeheader()
    writer.writerows(epoch_log)
print(f"에포크 로그 저장: {log_csv_path}")

log_md_lines = ["| epoch | train_loss | val AUC(CNN) | val AUC(ViT) | OOD AUC(CNN+ViT) |",
                "|---|---|---|---|---|"]
for row in epoch_log:
    ood_cell = f"{row['ood_auc_ensemble']:.4f}" if row["ood_auc_ensemble"] is not None else "-"
    log_md_lines.append(
        f"| {row['epoch']} | {row['train_loss']:.4f} | {row['val_auc_cnn']:.4f} | "
        f"{row['val_auc_vit']:.4f} | {ood_cell} |"
    )
log_md_path = os.path.join(CONFIG["output_dir"], "epoch_log.md")
with open(log_md_path, "w") as f:
    f.write("\n".join(log_md_lines) + "\n")
print(f"에포크 로그 마크다운 저장: {log_md_path}")

try:
    import matplotlib.pyplot as plt

    epochs_x = [r["epoch"] for r in epoch_log]
    val_auc_ens = [(r["val_auc_cnn"] + r["val_auc_vit"]) / 2 for r in epoch_log]
    ood_pts = [(r["epoch"], r["ood_auc_ensemble"]) for r in epoch_log if r["ood_auc_ensemble"] is not None]

    plt.figure(figsize=(7, 4.5))
    plt.plot(epochs_x, val_auc_ens, marker="o", label="In-distribution val AUC (CNN+ViT 평균)")
    if ood_pts:
        ox, oy = zip(*ood_pts)
        plt.plot(ox, oy, marker="s", label="OOD AUC (Gemini 등 held-out)")
    plt.axhline(0.5, color="gray", linestyle="--", linewidth=1, label="랜덤 수준 (AUC 0.5)")
    plt.xlabel("Epoch")
    plt.ylabel("AUC")
    plt.title("Epoch별 In-distribution vs OOD 일반화 성능")
    plt.ylim(0.4, 1.0)
    plt.legend()
    plt.grid(alpha=0.3)
    plot_path = os.path.join(CONFIG["output_dir"], "epoch_log.png")
    plt.savefig(plot_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"그래프 저장: {plot_path}")
except Exception as e:
    print(f"그래프 생성 실패 (치명적이지 않음, 표는 이미 저장됨): {e}")

try:
    from google.colab import files
    files.download(log_csv_path)
    files.download(log_md_path)
    _plot_path = os.path.join(CONFIG["output_dir"], "epoch_log.png")
    if os.path.exists(_plot_path):
        files.download(_plot_path)
except ImportError:
    pass

## 5. 가중치 저장 + 다운로드

다운로드된 `weights_cnn.pth`, `weights_vit.pth`를 로컬 `deepfakekiller/weights/` 폴더에 넣으면,
`deepfakekiller.py`가 실행 시 자동으로 불러옵니다 (코드 수정 불필요).

In [ ]:
import os

output_dir = CONFIG["output_dir"] if "CONFIG" in globals() else "/content/weights"
cnn_path = os.path.join(output_dir, "weights_cnn.pth")
vit_path = os.path.join(output_dir, "weights_vit.pth")

# 학습 루프가 매 epoch마다 이미 디스크에 저장해두므로, 메모리 변수가
# 세션 재시작으로 사라졌어도 파일이 남아있다면(커널만 재시작된 경우) 그대로 다운로드 가능
if not (os.path.exists(cnn_path) and os.path.exists(vit_path)):
    raise FileNotFoundError(
        "가중치 파일이 디스크에도 없습니다. 런타임이 완전히 새로 할당되어 /content가 초기화된 것이므로, "
        "처음(설정 셀)부터 다시 실행해서 재학습해야 합니다."
    )

print(f"가중치 파일: {cnn_path}, {vit_path}")

try:
    from google.colab import files
    files.download(cnn_path)
    files.download(vit_path)
except ImportError:
    print("Colab이 아닌 환경입니다. 파일 브라우저에서 직접 다운로드하세요:", output_dir)

## 6. 벤치마크 표 (AUC / EER, ablation)

`deepfakekiller.py`가 실제로 판정에 쓰는 세 신호(CNN, ViT, ELA)를 val split 전체에 대해
각각과 조합별로 채점해서 AUC(높을수록 좋음)와 EER(Equal Error Rate, 낮을수록 좋음) 표를 만든다.
로드맵에 있던 "결과표 없음" 항목을 채우는 셀 — README.md의 "결과표" 자리에 이 표를 그대로 붙여넣으면 된다.

**주의**: 이건 지금 학습에 쓴 데이터(ArtiFact ± 위에서 병합한 최신 생성기 데이터셋)의 **val split**에 대한
in-distribution 벤치마크다. "학습 때 전혀 안 쓴 별도 벤치마크(FaceForensics++/Celeb-DF 등)로 cross-dataset
일반화를 재는 것"은 로드맵에 아직 남아있는 별개 항목이다 — 이 표가 좋게 나와도 그 갭이 해소됐다는 뜻은 아니다.

In [ ]:
# ---------------------------------------------
# 6. 벤치마크 표 (AUC / EER) — CNN only / ViT only / CNN+ViT / +ELA
# ---------------------------------------------
# val_paths/val_labels(위 5번 셀에서 만든 val split) 전체에 대해 각 조합의 판별력을 측정한다.
# 가중치는 deepfakekiller.py의 compute_ai_score()와 동일하게 맞춤 (cnn 0.45 / vit 0.45 / ela 0.10).
import cv2
import numpy as np
from sklearn.metrics import roc_auc_score, roc_curve

ENSEMBLE_WEIGHTS = {"cnn": 0.45, "vit": 0.45, "ela": 0.10}


def calculate_ela_score(image_bgr, quality=90):
    """deepfakekiller.py의 calculate_ela_score()와 동일 로직 (독립 실행용으로 복사)."""
    try:
        _, encoded = cv2.imencode('.jpg', image_bgr, [cv2.IMWRITE_JPEG_QUALITY, quality])
        resaved = cv2.imdecode(encoded, cv2.IMREAD_COLOR)
        diff = cv2.absdiff(image_bgr, resaved)
        return min(np.mean(diff) / 255.0 * 10, 1.0)
    except Exception:
        return 0.5


def eer_from_scores(y_true, y_score):
    """Equal Error Rate: FPR와 FNR(=1-TPR)이 가장 가까워지는 지점의 평균."""
    fpr, tpr, _ = roc_curve(y_true, y_score)
    fnr = 1 - tpr
    idx = int(np.nanargmin(np.abs(fpr - fnr)))
    return float((fpr[idx] + fnr[idx]) / 2)


model_cnn.eval()
model_trans.eval()

y_true, s_cnn, s_vit, s_ela = [], [], [], []
with torch.no_grad():
    for path, label in zip(val_paths, val_labels):
        bgr = cv2.imread(path)
        if bgr is None:
            continue
        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        x = preprocess(Image.fromarray(rgb)).unsqueeze(0).to(device)

        s_cnn.append(torch.sigmoid(model_cnn(x)).item())
        s_vit.append(torch.sigmoid(model_trans(x)).item())
        s_ela.append(calculate_ela_score(bgr))
        y_true.append(int(label))

s_cnn, s_vit, s_ela = np.array(s_cnn), np.array(s_vit), np.array(s_ela)
s_ensemble = s_cnn * ENSEMBLE_WEIGHTS["cnn"] + s_vit * ENSEMBLE_WEIGHTS["vit"]
s_ensemble_ela = s_ensemble + s_ela * ENSEMBLE_WEIGHTS["ela"]

rows = [
    ("CNN only", s_cnn),
    ("ViT only", s_vit),
    ("CNN + ViT (0.45 / 0.45)", s_ensemble),
    ("CNN + ViT + ELA (deepfakekiller.py 기본값)", s_ensemble_ela),
]

table_lines = ["| 조합 | AUC | EER | n |", "|---|---|---|---|"]
for name, scores in rows:
    auc = roc_auc_score(y_true, scores)
    eer = eer_from_scores(y_true, scores)
    line = f"| {name} | {auc:.4f} | {eer:.4f} | {len(y_true)} |"
    print(line)
    table_lines.append(line)

benchmark_path = os.path.join(CONFIG["output_dir"], "benchmark.md")
with open(benchmark_path, "w") as f:
    f.write("\n".join(table_lines) + "\n")
print(f"\n마크다운 표 저장: {benchmark_path} — 이 내용을 README.md의 결과표 자리에 붙여넣으면 됨")

try:
    from google.colab import files
    files.download(benchmark_path)
except ImportError:
    pass

## 7. OOD 일반화 벤치마크 (Gemini 포함, 학습에 전혀 안 쓴 데이터)

위 6번 벤치마크는 "학습에 쓴 데이터의 val split"이라 in-distribution 성능만 보여준다.
이 셀은 `CONFIG["ood_eval_dataset"]`(기본값 `ComplexDataLab/OpenFake`의 `core/test`)로 —
**모델이 학습 중 단 한 번도 보지 않은** 생성기(Gemini의 nano-banana/nano-banana-pro 포함)에
대해 채점해서, README에 적힌 "최신 생성기 일반화 갭"이 재학습 후 실제로 줄었는지 검증한다.

**주의**: 이 데이터셋의 컬럼명(`image`/`label`)과 라벨 값(`"real"`/`"fake"` 문자열)은 실행 전
huggingface.co/datasets/ComplexDataLab/OpenFake 페이지에서 직접 한 번 확인해주세요 — 스키마가
바뀌었으면 설정 셀의 `ood_eval_*` 값들을 맞게 고치면 됩니다.

In [ ]:
# ---------------------------------------------
# 7. OOD 일반화 벤치마크 — 학습에 안 쓴 OpenFake core/test (Gemini nano-banana 포함)
# ---------------------------------------------
if not CONFIG.get("ood_eval_dataset"):
    print("CONFIG['ood_eval_dataset']가 비어 있어 OOD 벤치마크를 건너뜁니다.")
else:
    from datasets import load_dataset as _load_dataset

    print(f"OOD 평가셋 로드 중: {CONFIG['ood_eval_dataset']} "
          f"({CONFIG['ood_eval_config']}/{CONFIG['ood_eval_split']})")

    # streaming=True로 받는다: OpenFake 전체는 약 250만 장 규모라, 스트리밍 없이
    # split="test"만 받아도 실제로 쓸 max_samples(기본 1000개)보다 훨씬 많은 양을
    # 통째로 다운로드하게 된다. 스트리밍 + take()로 필요한 만큼만 내려받는다.
    # buffer_size는 200으로 작게 유지 — shuffle()이 디코딩된 이미지를 그 개수만큼
    # 메모리에 들고 있어서, 이전에 2000으로 뒀다가 4번 학습 루프 쪽에서 OOM으로
    # 세션이 죽은 적 있음(RAM 전부 사용).
    ood_ds = _load_dataset(
        CONFIG["ood_eval_dataset"],
        CONFIG["ood_eval_config"],
        split=CONFIG["ood_eval_split"],
        streaming=True,
    )

    cap_ood = CONFIG.get("ood_eval_max_samples")
    if cap_ood is not None:
        ood_ds = ood_ds.shuffle(seed=CONFIG["seed"], buffer_size=200).take(cap_ood)

    model_cnn.eval()
    model_trans.eval()

    ood_true, ood_s_cnn, ood_s_vit, ood_s_ela = [], [], [], []
    try:
        with torch.no_grad():
            for ex in iterate_with_timeout(ood_ds, CONFIG["stream_stall_timeout_sec"]):
                img = ex[CONFIG["ood_eval_image_col"]].convert("RGB")
                raw_label = ex[CONFIG["ood_eval_label_col"]]
                is_real = raw_label == CONFIG["ood_eval_real_value"]

                x = preprocess(img).unsqueeze(0).to(device)
                bgr = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)

                ood_s_cnn.append(torch.sigmoid(model_cnn(x)).item())
                ood_s_vit.append(torch.sigmoid(model_trans(x)).item())
                ood_s_ela.append(calculate_ela_score(bgr))
                ood_true.append(0 if is_real else 1)
    except Exception as e:
        # 큰 parquet 샤드가 스트리밍 도중 네트워크 오류로 끊기거나, 예외 없이 응답
        # 없이 멈출 수도 있다(그 경우 iterate_with_timeout이 강제로 끊음). 여기서
        # 안 잡으면 이 벤치마크 셀 전체가 죽으므로, 그때까지 받은 만큼만으로 계속 진행한다.
        print(f"⚠️ OOD 벤치마크 스트리밍 중 오류/멈춤으로 중단됨 ({e}). "
              f"지금까지 받은 {len(ood_true)}개로 계속 진행합니다.")

    print(f"OOD 평가 샘플 수: {len(ood_true)}")

    if not ood_true:
        print("OOD 평가 샘플을 하나도 못 받아서 벤치마크 표 생성을 건너뜁니다.")
    else:
        ood_s_cnn = np.array(ood_s_cnn)
        ood_s_vit = np.array(ood_s_vit)
        ood_s_ela = np.array(ood_s_ela)
        ood_ensemble = ood_s_cnn * ENSEMBLE_WEIGHTS["cnn"] + ood_s_vit * ENSEMBLE_WEIGHTS["vit"]
        ood_ensemble_ela = ood_ensemble + ood_s_ela * ENSEMBLE_WEIGHTS["ela"]

        ood_rows = [
            ("CNN only", ood_s_cnn),
            ("ViT only", ood_s_vit),
            ("CNN + ViT (0.45 / 0.45)", ood_ensemble),
            ("CNN + ViT + ELA (deepfakekiller.py 기본값)", ood_ensemble_ela),
        ]

        ood_table_lines = [
            f"### OOD 일반화 (`{CONFIG['ood_eval_dataset']}` {CONFIG['ood_eval_config']}/{CONFIG['ood_eval_split']}, 학습에 안 씀)",
            "| 조합 | AUC | EER | n |",
            "|---|---|---|---|",
        ]
        for name, scores in ood_rows:
            auc = roc_auc_score(ood_true, scores)
            eer = eer_from_scores(ood_true, scores)
            line = f"| {name} | {auc:.4f} | {eer:.4f} | {len(ood_true)} |"
            print(line)
            ood_table_lines.append(line)

        ood_benchmark_path = os.path.join(CONFIG["output_dir"], "benchmark_ood.md")
        with open(ood_benchmark_path, "w") as f:
            f.write("\n".join(ood_table_lines) + "\n")
        print(f"\n마크다운 표 저장: {ood_benchmark_path}")
        print("AUC가 6번 셀(in-distribution)보다 크게 낮다면 여전히 일반화 갭이 남아있다는 뜻,")
        print("6번과 비슷한 수준까지 올라왔다면 재학습으로 Gemini 등 최신 생성기 대응력이 실제로 좋아진 것.")

        try:
            from google.colab import files
            files.download(ood_benchmark_path)
        except ImportError:
            pass